# 02_risk_calibration.ipynb - Calibración de Csl y Verificación de BP

**Metodología**: `metodologia.pdf` Secciones 5 y 6

Este notebook implementa la calibración de riesgo en DOS PASOS SEPARADOS:

1. **Paso 1**: Calibración de Csl usando la MEDIANA de caídas implícitas (filtradas por precio + momentum alcista)
2. **Paso 2**: Verificación INDEPENDIENTE del Buying Power usando el Csl YA FIJO

> **IMPORTANTE**: Csl NUNCA se optimiza para lograr un BP objetivo. El BP se verifica DESPUÉS como filtro de admisión separado.

## Importaciones

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Ruta al repo
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Importar funciones implementadas
from src.risk import find_optimal_coef_sl, calculate_buying_power_distribution
from src.data import load_ohlc_from_yfinance

## Cargar Datos

**Ejecutar con un ticker real desde la lista `config/tickers_smallcap.txt`.**

In [16]:
# Cargar lista de tickers desde config/tickers_smallcap.txt
tickers_file = REPO_ROOT / 'config' / 'tickers_smallcap.txt'
if tickers_file.exists():
    tickers = [
        line.strip() for line in tickers_file.read_text().splitlines()
        if line.strip() and not line.strip().startswith('#')
    ]
else:
    tickers = ['ABSI', 'CRMD', 'FLGT', 'IMUX', 'KPTI', 'MCRB', 'NBTX', 'PRQR', 'SGMO']

print('Tickers disponibles (lista de ejemplo):', tickers)

# Selecciona el ticker que quieras analizar aquí
selected_ticker = 'ABSI'  # Cambia por cualquier ticker de la lista
print(f'Activo seleccionado: {selected_ticker}')

try:
    df = load_ohlc_from_yfinance(selected_ticker, period='max', interval='1h')
    print(f'Datos cargados desde yfinance para {selected_ticker}: {len(df)} barras')
except Exception as e:
    print(f'⚠️ No se pudo cargar {selected_ticker} desde yfinance: {e}')
    print('Usando datos sintéticos de demostración para mantener el notebook ejecutable.')
    np.random.seed(42)
    n_bars = 2000
    prices = 5 + 3 * np.cumsum(np.random.randn(n_bars)) / 100  # Precio pequeño con tendencia
    df = pd.DataFrame({
        'Datetime': pd.date_range('2023-01-01', periods=n_bars, freq='h'),
        'Open': prices * (1 + np.random.randn(n_bars) * 0.001),
        'High': prices * (1 + np.abs(np.random.randn(n_bars)) * 0.002),
        'Low': prices * (1 - np.abs(np.random.randn(n_bars)) * 0.002),
        'Close': prices,
        'Volume': np.random.randint(1000, 10000, n_bars)
    })

if 'Datetime' in df.columns:
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df = df.sort_values('Datetime').reset_index(drop=True)

print(f'Data loaded: {len(df)} bars')
print(f'Price range: ${df["Close"].min():.2f} - ${df["Close"].max():.2f}')
df.head()

Tickers disponibles (lista de ejemplo): ['ABSI', 'CRMD', 'FLGT', 'IMUX', 'KPTI', 'MCRB', 'NBTX', 'PRQR', 'SGMO', 'ASTE', 'BCYC', 'BCTX', 'BCRX', 'BEEM', 'CRBP', 'CRSR', 'EBON', 'EXTR', 'HBIO', 'HOTH', 'KROS', 'LRMR']
Activo seleccionado: ABSI
Datos cargados desde yfinance para ABSI: 3474 barras
Data loaded: 3474 bars
Price range: $2.06 - $11.32


,Datetime,Open,High,Low,Close,Volume
0,2024-06-27 09:30:00-04:00,3.090,3.200,3.030,3.055,235807
1,2024-06-27 10:30:00-04:00,3.055,3.060,2.995,3.050,235908
2,2024-06-27 11:30:00-04:00,3.055,3.055,3.010,3.045,124284
3,2024-06-27 12:30:00-04:00,3.050,3.070,3.010,3.040,172749
4,2024-06-27 13:30:00-04:00,3.040,3.060,3.030,3.055,110063


---

## Paso 1: Calibración de Csl (Mediana)

Según `metodologia.pdf` Sección 5:
- Filtrar entradas por precio small-cap ($1-$20) y momentum alcista (ER > 0.3)
- Calcular coeficiente implícito por entrada: `Csl_i = (P_entry - P_min_20) / ATR`
- Tomar la **MEDIANA** de los coeficientes (no promedio, no búsqueda por BP)
- Aplicar filtro de admisión: mediana debe estar en [1.5, 3.0]

In [17]:
c_sl = find_optimal_coef_sl(
    df=df,
    n_samples=500,
    lookforward_window=20,
    atr_period=50,
    price_min=1.0,
    price_max=20.0,
    momentum_k=10,
    momentum_threshold=0.3,
    admission_range=(1.5, 3.0),
    seed=42
)

if c_sl is None:
    print("⚠️  ACTIVO NO ADMITIDO: Datos insuficientes o Csl mediana fuera de rango [1.5, 3.0]")
else:
    print(f"✅ Csl calibrado (mediana): {c_sl:.3f}")

✅ Csl calibrado (mediana): 2.216


---

## Paso 2: Verificación de BP (Independiente)

Según `metodologia.pdf` Sección 6.2-6.3:
- Usar el Csl YA FIJO desde Paso 1
- Nueva muestra independiente de 500 entradas (semilla diferente: 7)
- Calcular distribución de Buying Power: `BP = P_entry * (risk_per_trade / (Csl * ATR))`
- Verificar: mediana(BP) ∈ [$100, $1800] Y max(BP) ≤ $3000

In [18]:
if c_sl is not None:
    bp_result = calculate_buying_power_distribution(
        df=df,
        c_sl=c_sl,
        n_samples=500,
        lookforward_window=20,
        atr_period=50,
        price_min=1.0,
        price_max=20.0,
        risk_per_trade=100.0,
        seed=7  # Semilla distinta de find_optimal_coef_sl
    )
    
    print(f"Buying Power Distribution (n={bp_result['n_valid_samples']} samples):")
    print(f"  - Mediana: ${bp_result['median_bp']:.2f}")
    print(f"  - Máximo:  ${bp_result['max_bp']:.2f}")
    print(f"  - Admitido: {'✅ YES' if bp_result['admitted'] else '❌ NO'}")
    
    if bp_result['admitted']:
        print(f"\n✅ ASSET APTO PARA TRADING con Csl = {c_sl:.3f}")
    else:
        print(f"\n⚠️  ACTIVO NO ADMITIDO: BP fuera de rango aceptable")
else:
    print("⏭️  Saltando Paso 2: Csl no se pudo calibrar en Paso 1")

Buying Power Distribution (n=500 samples):
  - Mediana: $1705.49
  - Máximo:  $2840.62
  - Admitido: ✅ YES

✅ ASSET APTO PARA TRADING con Csl = 2.216


---

## Resumen del flujo

```
Paso 1: find_optimal_coef_sl(df, ...) -> Csl_mediana
  ├─ Filtra: precio $1-$20 + ER > 0.3 (momentum alcista)
  ├─ Muestra: 500 entradas aleatorias
  ├─ Calcula: Csl_i = (P_entry - P_min_20) / ATR para cada entrada
  ├─ Toma: MEDIANA de Csl_i
  └─ Aprueba: si Csl_mediana ∈ [1.5, 3.0]

Paso 2: calculate_buying_power_distribution(df, c_sl=Csl_del_Paso1, ...
  ├─ Usa: Csl FIJO desde Paso 1 (NO optimizar)
  ├─ Muestra: 500 entradas NUEVAS (semilla diferente)
  ├─ Calcula: BP_i = P_entry * (risk_per_trade / (Csl * ATR))
  ├─ Verifica: mediana(BP) ∈ [$100, $1800] AND max(BP) ≤ $3000
  └─ Aprueba: si ambos criterios se cumplen
```


---

## Notas de implementación

1. **Ningún bucle de optimización BP**: Se eliminó completamente la "Fase 2" vieja que ordenaba Csl y buscaba el primero que cumpliera BP objetivo.

2. **Dos pasos separados**: Csl se calcula PRIMERO como mediana (sin mirar BP), luego BP se verifica DESPUÉS como filtro independiente.

3. **Semillas diferentes**: `find_optimal_coef_sl` usa `seed=42`, `calculate_buying_power_distribution` usa `seed=7` para muestras independientes.

4. **Admisión estricta**: Si cualquier paso falla, el activo se descarta (retorna None o admitted=False).